#Intialization

In [0]:
from pyspark.sql.functions import col,trim
from pyspark.sql.types import StringType,DataType
import pyspark.sql.functions as F

#Reading table from bronze

In [0]:
df = spark.table("workspace.bronze.crm_cust_info")
df.display()

# Silver Transformations

In [0]:
for field in df.schema.fields:
    if isinstance(field.dataType,StringType):
        df = df.withColumn(field.name,F.trim(F.col(field.name)))
df.display()


#Normalization

In [0]:
df =(
    df.withColumn(
        "cst_marital_status",
        F.when(F.upper(F.col("cst_marital_status")) == "S","Single")
         .when(F.upper(F.col("cst_marital_status")) == "M","Married")
         .otherwise("n/a")
    )
    .withColumn(
        "cst_gndr",
        F.when(F.upper(F.col("cst_gndr")) == "M","Male")
         .when(F.upper(F.col("cst_gndr"))== "F","Female")
         .otherwise("n/a")
    )
)

### ## # #Remove Records with Missing Customer ID

In [0]:
df = df.filter(F.col("cst_id").isNotNull())

### # Renaming columns

In [0]:
Rename_map ={
    "cst_id":"customer_id",
    "cst_key":"customer_number",
    "cst_firstname":"first_name",
    "cst_lastname":"last_name",
    "cst_marital_status":"martial_status",
    "cst_gndr":"gender",
    "cst_create_date":"created_date"
}

In [0]:
for old_name,new_name in Rename_map.items():
    df = df.withColumnRenamed(old_name,new_name) 

# sanity checks for dataframe

In [0]:
display(df.limit(10))

### Writing Silver Table

In [0]:
df.write.mode("overwrite").format("delta").saveAsTable("workspace.silver.crm_customers")

### #Sanity checks of silver table

In [0]:
%sql
select * from workspace.silver.crm_customers